<a href="https://colab.research.google.com/github/JVictorFD/JobMatchAI/blob/main/28_JV_Projeto_Final_JobMatch_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Final: JobMatch AI 🚀
**Sistema Inteligente de Recomendação de Vagas e Candidatos**

**Objetivos do Notebook:**
1. Unir as bases de dados (Vagas, Currículos e Skills).
2. Processar o texto com NLP (TF-IDF).
3. Treinar modelo de Classificação (Fit / No Fit).
4. Criar motor de Recomendação (Top-5 Vagas via Similaridade por Cosseno).
5. Treinar modelo de Regressão para Salário.

**Pipeline Automático de Dados:** Este notebook consome diretamente as bases de dados via API (`kagglehub` e HuggingFace Parquet) para NLP, Classificação e Recomendação.

In [1]:
# Instalação das bibliotecas necessárias para lidar com o KaggleHub e Parquet/HuggingFace
!pip install kagglehub pandas numpy scikit-learn pyarrow fastparquet fsspec -q

print("Bibliotecas instaladas com sucesso!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 6.4 MB/s eta 0:00:00
Bibliotecas instaladas com sucesso!


In [2]:
import pandas as pd
import numpy as np
import re
import os
import glob
import kagglehub

# NLP e ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


### 1. Ingestão de Dados (Data Ingestion) via API
Download e carregamento dinâmico dos datasets (LinkedIn, Currículos e Skills).

In [3]:
import kagglehub

# Este comando vai gerar um campo de texto no Colab
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [4]:
print("⏳ A iniciar o download das bases de dados...\n")

# --- 1. LinkedIn Job Postings (Kaggle) ---
print("1. A descarregar LinkedIn Job Postings...")
path_linkedin = kagglehub.dataset_download("arshkon/linkedin-job-postings")
# Procurar o primeiro ficheiro CSV dentro da pasta descarregada
csv_linkedin = glob.glob(os.path.join(path_linkedin, "*.csv"))[0]
df_vagas = pd.read_csv(csv_linkedin)
print(f"✅ Vagas carregadas! Total: {df_vagas.shape[0]} linhas.")

# --- 2. Resume-JD-Match (HuggingFace) ---
print("\n2. A descarregar Resume-JD-Match do HuggingFace...")
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_curriculos = pd.read_parquet("hf://datasets/facehuggerapoorv/resume-jd-match/" + splits["train"])
print(f"✅ Currículos carregados! Total: {df_curriculos.shape[0]} linhas.")

# --- 3. Job Skill Set Dataset (Kaggle) ---
print("\n3. A descarregar Job Skill Set...")
path_skills = kagglehub.dataset_download("batuhanmutlu/job-skill-set")
csv_skills = glob.glob(os.path.join(path_skills, "*.csv"))[0]
df_skills = pd.read_csv(csv_skills)
print(f"✅ Skills carregadas! Total: {df_skills.shape[0]} linhas.\n")

print("🎉 Todos os datasets foram carregados com sucesso para a memória RAM!")

⏳ A iniciar o download das bases de dados...

1. A descarregar LinkedIn Job Postings...


100%|██████████| 159M/159M [00:09<00:00, 17.9MB/s]

Extracting files...


✅ Vagas carregadas! Total: 123849 linhas.

2. A descarregar Resume-JD-Match do HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Currículos carregados! Total: 6241 linhas.

3. A descarregar Job Skill Set...


100%|██████████| 1.50M/1.50M [00:01<00:00, 1.50MB/s]

Extracting files...


✅ Skills carregadas! Total: 1167 linhas.

🎉 Todos os datasets foram carregados com sucesso para a memória RAM!


### 2. Processamento de Linguagem Natural (NLP)
Para que o nosso modelo compreenda os currículos e as vagas, precisamos de limpar os textos (remover pontuações, números) e transformá-los em matrizes numéricas através da técnica **TF-IDF**.

In [5]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

print("A inspecionar as colunas do dataset de currículos...")
# Ajuste manual baseado na estrutura real do dataset (colunas: 'text', 'label')
if 'text' in df_curriculos.columns:
    col_texto = 'text'
    col_target = 'label'
else:
    # Fallback caso as colunas mudem
    col_texto = df_curriculos.columns[0]
    col_target = df_curriculos.columns[-1]

def limpar_texto(texto):
    if pd.isna(texto): return ""
    texto = str(texto).lower()
    # Mantém apenas letras e espaços
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    # Remove espaços duplos
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

print("A limpar os textos...")
df_curriculos['text_clean'] = df_curriculos[col_texto].apply(limpar_texto)

print("A converter texto em números (TF-IDF)...")
# Usamos max_features=5000 para eficiência
tfidf_clf = TfidfVectorizer(stop_words='english', max_features=5000)

X = tfidf_clf.fit_transform(df_curriculos['text_clean'])

# Converter labels de texto ('Good Fit' / 'No Fit') para números
y = df_curriculos[col_target].map({'Good Fit': 1, 'No Fit': 0, 'Match': 1, 'No Match': 0})
# Se houver valores nulos após o map (caso os labels sejam diferentes), preenchemos com 0
y = y.fillna(0).astype(int)

print(f"✅ Matriz TF-IDF pronta! Tamanho: {X.shape}")
print(f"✅ Labels processados. Distribuição: {y.value_counts().to_dict()}")

A inspecionar as colunas do dataset de currículos...
A limpar os textos...
A converter texto em números (TF-IDF)...
✅ Matriz TF-IDF pronta! Tamanho: (6241, 5000)
✅ Labels processados. Distribuição: {0: 4699, 1: 1542}


### 4. Motor de Recomendação de Vagas (LinkedIn)
Agora vamos criar a funcionalidade principal do JobMatch AI: receber um currículo e buscar nas vagas do LinkedIn as Top-5 que melhor combinam com o perfil usando **Similaridade por Cosseno**.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# Localizar dinamicamente a coluna de descrição e título nas vagas do LinkedIn
cols_vagas = [c.lower() for c in df_vagas.columns]
col_vaga_desc = 'description' if 'description' in cols_vagas else df_vagas.columns[1]
col_vaga_title = 'title' if 'title' in cols_vagas else df_vagas.columns[0]

def recomendar_vagas(meu_curriculo, df_base_vagas, top_n=5):
    print("A analisar o seu perfil...")

    cv_limpo = limpar_texto(meu_curriculo)
    vagas_limpas = df_base_vagas[col_vaga_desc].apply(limpar_texto)

    # Vetorizador independente para a recomendação
    tfidf_rec = TfidfVectorizer(stop_words='english', max_features=5000)

    # Junta o CV como o primeiro item, seguido das vagas
    todos_textos = [cv_limpo] + vagas_limpas.tolist()
    matriz_tfidf = tfidf_rec.fit_transform(todos_textos)

    # Separa o CV das vagas
    vetor_cv = matriz_tfidf[0]
    vetores_vagas = matriz_tfidf[1:]

    # Calcula o "Score de Aderência" (0 a 1)
    similaridades = cosine_similarity(vetor_cv, vetores_vagas).flatten()

    # Pega os índices dos maiores scores
    top_indices = similaridades.argsort()[-top_n:][::-1]

    print("\n🚀 --- JOBMATCH AI: AS SUAS TOP VAGAS RECOMENDADAS --- 🚀")
    for rank, idx in enumerate(top_indices, 1):
        titulo = df_base_vagas.iloc[idx][col_vaga_title]
        score = similaridades[idx] * 100
        print(f"{rank}. {titulo} | Match: {score:.1f}%")

# =========================================
# TESTE DO SISTEMA
# =========================================
# Altere este texto para testar diferentes profissões!
meu_perfil = """
Data Scientist with Machine Learning
I am a Software Engineer with 5 years of experience.
I have strong skills in Python, Django, SQL, and AWS.
I like building backend APIs and working with databases.
"""

# ⚠️ Atenção: Para evitar travamentos no Colab gratuito, vamos fazer a pesquisa
# nas primeiras 5.000 vagas. Se quiser, pode aumentar esse número depois!
amostra_vagas = df_vagas.head(50000)

recomendar_vagas(meu_perfil, amostra_vagas)

A analisar o seu perfil...

🚀 --- JOBMATCH AI: AS SUAS TOP VAGAS RECOMENDADAS --- 🚀
1. GoLang Developer (Capital One Experience Required) - ***W2 Only*** | Match: 45.1%
2. Contract Developer | Match: 41.3%
3. Java Application Developer | Match: 39.7%
4. backend Java developer | Match: 36.8%
5. Cloud Engineer | Match: 35.7%


### 5. Predição de Salário (Regressão)
Usando as vagas que possuem salário divulgado no LinkedIn, vamos treinar um `RandomForestRegressor` para prever a estimativa salarial de novas vagas baseando-se na descrição do cargo.

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

print("A procurar vagas com salário divulgado na base do LinkedIn...")

# Identificar a coluna de salário (lidando com diferentes versões do dataset)
cols_linkedin = [c.lower() for c in df_vagas.columns]
col_salario = 'med_salary' if 'med_salary' in cols_linkedin else ('max_salary' if 'max_salary' in cols_linkedin else None)

if col_salario is None:
    print("Aviso: Nenhuma coluna de salário exata encontrada. A usar uma simulação para efeitos didáticos.")
    # Apenas como salvaguarda se o dataset for diferente
    df_vagas['target_salary'] = np.random.randint(40000, 150000, df_vagas.shape[0])
    col_salario = 'target_salary'

# Filtrar apenas as vagas que têm salário preenchido (não nulo)
df_com_salario = df_vagas.dropna(subset=[col_salario]).copy()

# Vamos pegar numa amostra (ex: 3000 vagas) para não sobrecarregar a memória
df_com_salario = df_com_salario.head(3000)
print(f"Treinando com {df_com_salario.shape[0]} vagas que possuem salário preenchido...")

# 1. Transformar o texto das vagas em números
tfidf_reg = TfidfVectorizer(stop_words='english', max_features=3000)
X_salario = tfidf_reg.fit_transform(df_com_salario[col_vaga_desc].apply(limpar_texto))
y_salario = df_com_salario[col_salario]

# 2. Dividir em Treino e Teste
X_tr, X_te, y_tr, y_te = train_test_split(X_salario, y_salario, test_size=0.2, random_state=42)

# 3. Treinar o Modelo de Regressão
print("A treinar o Random Forest Regressor (Pode demorar uns segundos)...")
regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_tr, y_tr)

# 4. Avaliar o modelo
y_pred_salario = regressor.predict(X_te)
erro_medio = mean_absolute_error(y_te, y_pred_salario)

print("\n💵 --- RESULTADO DO MODELO DE SALÁRIO --- 💵")
print(f"Erro Médio Absoluto (MAE): $ {erro_medio:.2f} por ano")
print(f"R² Score (Precisão): {r2_score(y_te, y_pred_salario):.2f}")

# --- Testando a predição salarial numa vaga inventada ---
vaga_inventada = "Senior Python Developer with 10 years experience in AWS, Django, and Machine Learning."
vaga_vetor = tfidf_reg.transform([limpar_texto(vaga_inventada)])
salario_estimado = regressor.predict(vaga_vetor)[0]

print(f"\nSalário estimado para a Vaga Inventada: $ {salario_estimado:,.2f}")

A procurar vagas com salário divulgado na base do LinkedIn...
Treinando com 3000 vagas que possuem salário preenchido...
A treinar o Random Forest Regressor (Pode demorar uns segundos)...

💵 --- RESULTADO DO MODELO DE SALÁRIO --- 💵
Erro Médio Absoluto (MAE): $ 20021.37 por ano
R² Score (Precisão): 0.35

Salário estimado para a Vaga Inventada: $ 8,086.72


### 3. Modelo de Classificação (Fit vs No Fit)
Como a nossa base de dados está desbalanceada (muito mais exemplos "No Fit" do que "Fit"), utilizaremos o parâmetro `class_weight='balanced'` na Regressão Logística para penalizar mais os erros cometidos na classe minoritária.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Divisão de Treino e Teste (80% treino, 20% teste)
# O stratify=y é crucial aqui por causa do desbalanceamento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("A treinar a Inteligência Artificial (Regressão Logística)...")
# max_iter=1000 garante que o modelo tem tempo suficiente para aprender
modelo_fit = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo_fit.fit(X_train, y_train)

# 2. Avaliação na base de teste
y_pred = modelo_fit.predict(X_test)

print("\n🎯 --- RESULTADO DO MODELO DE CLASSIFICAÇÃO --- 🎯")
# classification_report mostra a Precisão, Recall e F1-Score
print(classification_report(y_test, y_pred))

A treinar a Inteligência Artificial (Regressão Logística)...

🎯 --- RESULTADO DO MODELO DE CLASSIFICAÇÃO --- 🎯
              precision    recall  f1-score   support

           0       0.93      0.79      0.85       940
           1       0.56      0.81      0.66       309

    accuracy                           0.80      1249
   macro avg       0.74      0.80      0.76      1249
weighted avg       0.84      0.80      0.81      1249



---
##Gerando aplicação web interativa - teste

In [9]:
# Empacotando os dados gerados pelas manipulações e trinamentos em um .pkl
import joblib

print("A guardar os modelos treinados...")

# Guardar o classificador (Fit/No Fit) e o seu vetorizador
joblib.dump(modelo_fit, 'modelo_fit.pkl')
joblib.dump(tfidf_clf, 'tfidf_clf.pkl')

# Guardar o regressor (Salários) e o seu vetorizador
joblib.dump(regressor, 'modelo_salario.pkl')
joblib.dump(tfidf_reg, 'tfidf_reg.pkl')

# Guardar uma amostra de 5.000 vagas para a app consultar rapidamente
df_vagas.head(5000).to_csv('vagas_app.csv', index=False)

print("✅ Ficheiros exportados com sucesso! O seu Cérebro está pronto para a App.")

A guardar os modelos treinados...
✅ Ficheiros exportados com sucesso! O seu Cérebro está pronto para a App.


##Criar o ficheiro da Aplicação Web (app.py)

In [10]:
# Criar ficheiro app
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Configuração da Página
st.set_page_config(page_title="JobMatch AI", page_icon="🚀", layout="wide")

# 2. Função de Limpeza
def limpar_texto(texto):
    if pd.isna(texto): return ""
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# 3. Carregar os modelos (com cache para não recarregar a cada clique)
@st.cache_resource
def carregar_dados():
    modelo_fit = joblib.load('modelo_fit.pkl')
    tfidf_clf = joblib.load('tfidf_clf.pkl')
    regressor = joblib.load('modelo_salario.pkl')
    tfidf_reg = joblib.load('tfidf_reg.pkl')
    df_vagas = pd.read_csv('vagas_app.csv')
    return modelo_fit, tfidf_clf, regressor, tfidf_reg, df_vagas

# 4. Interface Web
st.title("🎯 JobMatch AI")
st.subheader("O seu Sistema de Recomendação Inteligente de Vagas")
st.write("Insira o seu perfil profissional abaixo e descubra as melhores vagas para si, com análise de Fit e estimativa salarial.")

try:
    modelo_fit, tfidf_clf, regressor, tfidf_reg, df_vagas = carregar_dados()

    # Encontrar as colunas corretas dinamicamente
    cols_vagas = [c.lower() for c in df_vagas.columns]
    col_vaga_desc = 'description' if 'description' in cols_vagas else df_vagas.columns[1]
    col_vaga_title = 'title' if 'title' in cols_vagas else df_vagas.columns[0]

except FileNotFoundError:
    st.error("Modelos não encontrados. Por favor, execute as células de treino primeiro.")
    st.stop()

# Caixa de texto para o utilizador
curriculo = st.text_area("Cole o seu Resumo Profissional / Currículo aqui:", height=150)

# Botão Mágico
if st.button("Analisar o Meu Perfil 🚀"):
    if curriculo.strip() == "":
        st.warning("Por favor, escreva alguma coisa no currículo!")
    else:
        with st.spinner('A IA está a analisar o seu perfil face ao mercado...'):
            cv_limpo = limpar_texto(curriculo)

            # Limpar vagas
            vagas_limpas = df_vagas[col_vaga_desc].apply(limpar_texto)

            # Calcular Similaridade
            tfidf_rec = TfidfVectorizer(stop_words='english', max_features=5000)
            textos_totais = [cv_limpo] + vagas_limpas.tolist()
            matriz_tfidf = tfidf_rec.fit_transform(textos_totais)

            scores = cosine_similarity(matriz_tfidf[0], matriz_tfidf[1:]).flatten()
            top_indices = scores.argsort()[-5:][::-1] # Top 5

            st.markdown("---")
            st.header("🏆 As Suas Top 5 Vagas")

            # Mostrar os resultados de forma elegante
            for i, idx in enumerate(top_indices, 1):
                vaga = df_vagas.iloc[idx]
                score_match = scores[idx] * 100
                vaga_desc_limpa = vagas_limpas.iloc[idx]

                # Prever Fit/No Fit
                texto_fit = cv_limpo + " " + vaga_desc_limpa
                vetor_fit = tfidf_clf.transform([texto_fit])
                is_fit = modelo_fit.predict(vetor_fit)[0]

                # Prever Salário
                vetor_salario = tfidf_reg.transform([vaga_desc_limpa])
                salario_estimado = regressor.predict(vetor_salario)[0]

                # Formatação Visual
                status_fit = "✅ FIT EXCELENTE" if is_fit == 1 else "⚠️ FIT PARCIAL (Requer Skills Adicionais)"
                cor_fit = "green" if is_fit == 1 else "orange"

                with st.expander(f"{i}. {vaga[col_vaga_title]}  |  Match: {score_match:.1f}%"):
                    st.markdown(f"**Análise da IA:** :{cor_fit}[{status_fit}]")
                    st.write(f"**Salário Estimado:** $ {salario_estimado:,.2f} / ano")
                    st.write(f"**Descrição da Vaga:** {str(vaga[col_vaga_desc])[:600]}...")

Writing app.py


##Colocar a App Online para teste com localtunnel

In [15]:
import urllib

# 1. Forçamos a instalação de uma versão do Streamlit que funciona 100% com Localtunnel
!pip install -q streamlit==1.25.0
!npm install localtunnel -q

# 2. Obter a "Password" para o site
ip_password = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"\n=======================================================")
print(f"👉 A SUA PASSWORD (IP) É: {ip_password}")
print(f"=======================================================\n")

# 3. Correr a aplicação desativando o CORS e XSRF (que causam o erro de Fetch)
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧
up to date, audited 23 packages in 1s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠇
👉 A SUA PASSWORD (IP) É: 35.229.194.210

⠙⠹

your url is: https://grumpy-words-shake.loca.lt

  You can now view your Streamlit app in your browser.

  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.194.210:8501

  Stopping...
^C


##Como usar o Link Público:
###Ao rodar o Passo 3, vai aparecer a sua "Password (IP)" no ecrã e, logo abaixo, um link terminado em loca.lt (ex: https://alguma-coisa.loca.lt).

* Clique nesse link.

* Na página que abrir, irá pedir um "Endpoint IP" (ou Password). Cole aí o número do IP que apareceu no Passo 3 e clique em Submit.

* Magia! 🎉 O seu sistema JobMatch AI completo vai aparecer no ecrã, pronto a ser usado por si ou pelos seus professores!

---

In [13]:
# 1. Certifique-se de que o Streamlit corre em segundo plano na porta padrão (8501)
!streamlit run app.py &

# 2. Ative o encaminhamento de porta seguro do Colab
from google.colab import output
output.serve_kernel_port_as_iframe(8501)




  You can now view your Streamlit app in your browser.

  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.194.210:8501

  Stopping...


<IPython.core.display.Javascript object>

## Criar o ficheiro requeriments.txt

In [14]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
joblib

Writing requirements.txt
